# Load/Save using both pandas and huggingface <mark> lines=True critical in pandas

In [1]:
import pandas as pd
#pandas
#load
tst=pd.read_json("tst.json", orient="records",lines=True)
trn=pd.read_json("trn.json", orient="records",lines=True)
eval=pd.read_json("eval.json", orient="records",lines=True)

#save
trn.to_json("trn.json", orient="records",lines=True)
eval.to_json("eval.json", orient="records",lines=True)
tst.to_json("tst.json", orient="records",lines=True)
# eval.head()

In [2]:
#huggingface datasets

#load
from datasets import load_dataset
train_dataset = load_dataset("json", data_files="trn.json",split="train") #gets the whole thing
eval_dataset = load_dataset("json", data_files="eval.json", split="train")
tst_dataset = load_dataset("json", data_files="tst.json", split="train")

#save it
train_dataset.to_json(f"trn.json",orient='records',lines=True)
eval_dataset.to_json(f"eval.json",orient='records',lines=True)
tst_dataset.to_json(f"tst.json",orient='records',lines=True)

/home/kperkins411/anaconda3/envs/p311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating train split: 2948 examples [00:00, 195457.05 examples/s]
Generating train split: 400 examples [00:00, 156533.08 examples/s]
Generating train split: 1019 examples [00:00, 225790.89 examples/s]
Creating json from Arrow format: 100%|██████████| 2/2 [00:00<00:00, 298.40ba/s]


843762

In [48]:
# td = load_dataset("json", data_files="trn.json",split=None) #gets a dict you get to it with ['train']
# td['train'][0]

# td = load_dataset("json", data_files="trn.json",split="train")
# td[0]

{'positive': 'Information We Collect From Other Sources We may also receive information from other sources and combine that with information we collect through our Services. For example: If you choose to link, create, or log in to your Uber account with a payment provider (e.g., Google Wallet) or social media service (e.g., Facebook), or if you engage with a separate app or website that uses our API (or whose API we use), we may receive information about you or your connections from that site or app.',
 'anchor': 'What safeguards are in place to protect the information obtained from third-party sources?',
 'id': 0}

# huggingface dataset column manipulation

In [3]:
import datasets

#rename columns
tds = train_dataset.rename_column("anchor", "query")
tds = train_dataset.rename_column("positive", "pos")

#remove unneeded columns
# tds=tds.remove_columns(["id",'most_dissimilar_context'])

#add a new column that consists of empty lists
new_column = [ [] for _ in range(len(train_dataset)) ]
tds=train_dataset.add_column('neg',new_column)

# Column conversion and position manipulation

In [4]:
import os
import sys
PROJECT_ROOT = os.path.abspath(os.path.join(
 os.getcwd(),
 os.pardir+'/code')
)
#only add it once
if (PROJECT_ROOT not in sys.path):
 sys.path.append(PROJECT_ROOT)

import pandas as pd
import utils

#convert "position" column to list
tds=utils.change_col_to_list(train_dataset,'positive')

Token is valid (permission: write).
Your token has been saved in your configured git credential helpers (store).
Your token has been saved to /home/kperkins411/.cache/huggingface/token
Login successful


In [5]:
train_dataset[0]

{'anchor': 'What safeguards are in place to protect the information obtained from third-party sources?',
 'positive': 'Information We Collect From Other Sources We may also receive information from other sources and combine that with information we collect through our Services. For example: If you choose to link, create, or log in to your Uber account with a payment provider (e.g., Google Wallet) or social media service (e.g., Facebook), or if you engage with a separate app or website that uses our API (or whose API we use), we may receive information about you or your connections from that site or app.',
 'id': 0}

# slicing

In [6]:
#select just 10 rows
tds=train_dataset.select(range(10))
tds

Dataset({
    features: ['anchor', 'positive', 'id'],
    num_rows: 10
})

In [7]:
print(train_dataset.features)
print(train_dataset)
train_dataset[1]['positive']

{'anchor': Value(dtype='string', id=None), 'positive': Value(dtype='string', id=None), 'id': Value(dtype='int64', id=None)}
Dataset({
    features: ['anchor', 'positive', 'id'],
    num_rows: 2948
})


'Each of the Suppliers warrants that the Products shall comply with the specifications and documentation agreed by the relevant Supplier and the Company in writing that is applicable to such Products for the Warranty Period.'

In [1]:
from sentence_transformers.util import mine_hard_negatives

/home/kperkins411/sentence-transformers/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


In [1]:
# mine_hard_negatives??


# convert pandas to dataset and dataset to pandas

In [6]:
import datasets
from datasets import Dataset
from datasets import load_dataset
import pandas as pd
eval_pd=pd.read_json("eval.json", orient="records",lines=True) #pandas

eval_hf = datasets.Dataset.from_pandas(eval_pd, preserve_index=False)  #from pandas to huggingface
print(eval_hf)
eval_pd2=pd.DataFrame(eval_hf) #from huggingface to pandas
eval_pd2.equals(eval_pd)


Dataset({
    features: ['anchor', 'positive', 'id'],
    num_rows: 400
})


True